# Stage 6: three checks on the development year

**Reads:** `data/gravity_clean.csv`, saved by stage 2; `results/dev_runs.csv` and `results/dev_components.csv`, saved by stage 3; `results/size_rule.csv`, `results/gaps.csv` and `results/cutoff.csv`, saved by stage 4.

**Writes:**
- `results/check_runs.csv`: one row per configuration and training draw of each check, with the columns of `dev_runs.csv`. The column `variant` names the check.
- `results/check_components.csv`: for the checks `noise` and `split2`, one row per training size and configuration with the mean test deviance split into variance and remainder, with the columns of `dev_components.csv`.
- `results/check_cutoff.csv`: for `noise` and `split2`, the pattern and n* of each comparison with their bootstrap, with the columns of `cutoff.csv` after `variant`.

All three are read by stage 7.

The three checks of stage 6 in section 6 of `docs/build_plan.md`, all on the development year and each run with the functions of stages 3 and 4. None reads the analysis year, and none changes anything that decision 21 froze.

- `noise`: the base experiment of stage 3 again, on the same training samples with the same seeds, with every training flow multiplied by its own random factor of mean 1 and variance `NOISE_VAR` (decision 5). The expected flow stays the same and its spread grows. The test flows are not touched. This is the plan's check on the mechanism: the bias-variance argument says that noise should hurt the flexible tree ensembles more than the gravity equation.
- `split2`: the base experiment again with another random split of the country pairs into training pool and test set (split 1 instead of split 0), and so other test pairs and other training samples. It shows how much of the development-year pattern depends on the split.
- `trees2x`: at the largest size only, on the training samples of the base run, the tuned forest with twice the trees and the tuned boosting with twice the largest number of trees. It checks whether the ensembles were held back by too few trees.

In `noise` and `split2`, `rf` and `gbm` are tuned on the experiment's own test set, as stage 4 tuned them in the base experiment, so each check is read the way the base results were, and like them it is exploratory. The ridge line is fitted on every draw of `noise` and `split2`, as `run_draw` always does, and its rows are in the files, but this stage reports nothing on it (decision 18). Every standard error and interval here describes variation over training draws from one pool, scored on one test set. The draws overlap, more so at large sizes.

*Drafted with Claude (Anthropic), which was asked for stage 6 of the build plan after Yu's run of stage 5: the checks `noise`, `split2` and `trees2x` on the development year with the functions of stages 3 and 4, their values printed next to those of the base run, and the three files stage 7 reads. It replaces the placeholder of 18 October 2026.*

In [ ]:
from helpers import (BOOTSTRAP, CHECKS, COMPARISONS, DATA, DEV_YEAR, DRAWS, GBM_MAX_TREES, N_JOBS, NOISE_VAR, PATTERNS,
                     PREDICTORS, RESULTS, RF_TREES, ROOT, SIZES, TARGET, check_find_crossing, draw_rows,
                     experiment_cutoff, frozen_settings, largest_relative_difference, noise_factor, projected_seconds,
                     refuse_part, run_draw, run_experiment, run_trees2x_draw, run_trees2x_draws, runs_table,
                     size_rule_lookup, split_pairs, summarise, test_groups, trees2x_counts, trees2x_differences,
                     trees2x_rows, tune)

import time

import pandas as pd

refuse_part()
# Section 8: find_crossing on toy gaps, before it is used on real ones.
check_find_crossing()

# The base experiment of stages 3 and 4 uses split 0, and so do noise and
# trees2x; split2 uses split 1.
BASE_SPLIT = 0
SECOND_SPLIT = 1
largest = max(SIZES)
print(f"Development year {DEV_YEAR}; sizes {list(SIZES)}; {DRAWS} draws per size; {N_JOBS} worker(s)")
print(f"Checks: {', '.join(CHECKS)}")

## The base results

Each check is compared with the base experiment of the development year, as stages 3 and 4 saved it. The files must be those of the full run: only the base experiment, every size of `SIZES` and `DRAWS` draws for each size and configuration. The settings in `size_rule.csv`, which the check `trees2x` starts from, must be those that `tune` finds again on `dev_runs.csv`.

In [ ]:
# round_trip reads every number back exactly as stages 3 and 4 wrote it.
runs_base = pd.read_csv(RESULTS / "dev_runs.csv", float_precision="round_trip")
components_base = pd.read_csv(RESULTS / "dev_components.csv", float_precision="round_trip")
rule = pd.read_csv(RESULTS / "size_rule.csv", float_precision="round_trip")
gaps_base = pd.read_csv(RESULTS / "gaps.csv", float_precision="round_trip")
cutoff_base = pd.read_csv(RESULTS / "cutoff.csv", float_precision="round_trip")

for table in (runs_base, components_base):
    assert (table["variant"] == "base").all() and (table["year"] == DEV_YEAR).all() and \
        (table["split"] == BASE_SPLIT).all(), "The files of stage 3 hold rows other than the base experiment."
sizes = sorted(int(n) for n in runs_base["size"].unique())
assert sizes == list(SIZES), f"dev_runs.csv has the sizes {sizes}, not SIZES {list(SIZES)}: not the full run of stage 3."
draws_per = runs_base.groupby(["size", "config"])["draw"].nunique()
assert (draws_per == DRAWS).all(), f"dev_runs.csv does not have {DRAWS} draws for every size and configuration."

tuned_base = tune(summarise(runs_base))
for row in tuned_base.itertuples():
    in_rule = size_rule_lookup(rule, row.size)
    assert in_rule.loc[in_rule["model"] == row.model, "config"].item() == row.config, \
        f"size_rule.csv does not have the {row.model} that tune finds on dev_runs.csv at size {row.size}."
print(f"dev_runs.csv: {len(runs_base):,} rows; dev_components.csv: {len(components_base):,} rows; "
      f"{DRAWS} draws for each of {len(draws_per):,} sizes and configurations")
print("The tuned rf and gbm of size_rule.csv are those tune finds on dev_runs.csv at every size.")

## The training pool and the test set of each split

As in stage 3, a random quarter of the unordered country pairs goes to the test set with all its rows (`split_pairs`), so both directions of a pair land on the same side. Split 0 is the split of the base run, used by `noise` and `trees2x`; split 1 is drawn with another key and used by `split2`. The largest size has to be at most a quarter of each pool. If it is not, the stage stops here: the size is not shrunk silently.

In [ ]:
# Country identifiers are text, as in stage 2: no value is read as missing.
table = pd.read_csv(DATA / "gravity_clean.csv", dtype={"country_id_o": str, "country_id_d": str, "pair_id": str},
                    keep_default_na=False, na_values=[""])
print(f"{DEV_YEAR}: {(table['year'] == DEV_YEAR).sum():,} rows")
splits, test_pairs = {}, {}
for split_id in (BASE_SPLIT, SECOND_SPLIT):
    pool, test = split_pairs(table, DEV_YEAR, split_id)
    print(f"Split {split_id}: training pool {len(pool):,} rows, {pool['pair_id'].nunique():,} pairs; "
          f"test set {len(test):,} rows, {test['pair_id'].nunique():,} pairs")
    assert largest <= len(pool) / 4, \
        f"The largest size, {largest:,}, is over a quarter of the pool of split {split_id} ({len(pool):,} rows). Stop and report."
    test_pairs[split_id] = set(test["pair_id"])
    # The ridge line keeps both directions of a pair in one fold of its cross-validation.
    splits[split_id] = {"X_pool": pool[PREDICTORS].to_numpy(dtype=float), "y_pool": pool[TARGET].to_numpy(dtype=float),
                        "pair_pool": pool["pair_id"].to_numpy(), "X_test": test[PREDICTORS].to_numpy(dtype=float),
                        "y_test": test[TARGET].to_numpy(dtype=float)}
shared = len(test_pairs[BASE_SPLIT] & test_pairs[SECOND_SPLIT])
print(f"Test pairs in both splits: {shared:,} of {len(test_pairs[SECOND_SPLIT]):,}")

## Check: the base run comes back

Before any check is run, draw 0 at the smallest size is fitted again on split 0 with `run_draw`, without noise. Its rows must equal those of `dev_runs.csv` apart from the seconds, within a relative difference of 1e-9. This shows that the pool, the test set, the training samples and the seeds here are those of the base run, which `noise` and `trees2x` reuse.

In [ ]:
base = splits[BASE_SPLIT]
n = min(SIZES)
result = run_draw(base["X_pool"], base["y_pool"], base["pair_pool"], base["X_test"], base["y_test"], DEV_YEAR,
                  BASE_SPLIT, n, 0)
keys = {"variant": "base", "year": DEV_YEAR, "split": BASE_SPLIT, "size": n, "draw": 0}
rerun = runs_table(draw_rows(result, base["y_test"], test_groups(base["y_test"]), keys))
difference, shared = largest_relative_difference(rerun, runs_base, ["variant", "year", "split", "size", "draw", "config"])
assert shared == len(rerun), f"{len(rerun) - shared} row(s) of the draw are not in dev_runs.csv."
assert difference <= 1e-9, f"The draw differs from dev_runs.csv by {difference:.1e}. Stop and report."
print(f"n = {n}, draw 0: {shared} rows, largest relative difference from dev_runs.csv {difference:.1e}")

## Check: the noise factor

In `noise`, each training flow is multiplied by its own factor exp(s z - s²/2), where z is standard normal and s² = ln(1 + `NOISE_VAR`) (`noise_factor`). Its mean is 1 and its variance `NOISE_VAR`. A million factors from the generator of the check, keyed by a size of 0 that no draw uses, must show this: a mean within 0.01 of 1 and a variance within 0.05 of `NOISE_VAR`. Both margins are several times the standard error of a million draws, so the check fails only if the formula is wrong.

In [ ]:
factors = noise_factor(1_000_000, DEV_YEAR, BASE_SPLIT, 0, 0)
print(f"A million factors: mean {factors.mean():.4f}, variance {factors.var():.4f} (NOISE_VAR = {NOISE_VAR})")
assert abs(factors.mean() - 1) <= 0.01 and abs(factors.var() - NOISE_VAR) <= 0.05, \
    "The noise factor does not have mean 1 and variance NOISE_VAR."

## Time one draw of each kind

One draw of `noise` and one draw of `trees2x` at the largest size are timed, and the minutes of the stage are projected from them: the draws of `noise` and of `split2` at every size, with the ratios between sizes measured in section 9 of the plan, and the draws of `trees2x`. The bootstrap is not counted. Only seconds are printed here, no error.

In [ ]:
settings = frozen_settings(size_rule_lookup(rule, largest))
start = time.perf_counter()
run_draw(base["X_pool"], base["y_pool"], base["pair_pool"], base["X_test"], base["y_test"], DEV_YEAR, BASE_SPLIT,
         largest, 0, noise=True)
experiment_seconds = time.perf_counter() - start
start = time.perf_counter()
run_trees2x_draw(base["X_pool"], base["y_pool"], base["X_test"], DEV_YEAR, BASE_SPLIT, largest, 0, settings["rf"],
                 settings["gbm"])
trees2x_seconds = time.perf_counter() - start
projected = (2 * sum(DRAWS * projected_seconds(experiment_seconds, largest, n) for n in SIZES)
             + DRAWS * trees2x_seconds) / 60
print(f"One draw at n = {largest:,}: {experiment_seconds:.1f} seconds with noise, {trees2x_seconds:.1f} seconds of trees2x")
print(f"Projected time of stage 6: {projected:.0f} minutes on one worker"
      + (f", about {projected / N_JOBS:.0f} with {N_JOBS} workers" if N_JOBS > 1 else ""))

## The check `noise`

The base experiment of stage 3 (`run_experiment`, the same function stage 3 runs), with `noise=True`: on every draw the training flows are multiplied by factors from `rng("noise", year, split, size, draw)` before any model sees them. The training sample, the seeds and the test set are those of the base run. Everything computed on the training sample uses the noisy flows: the training deviance, the largest training flow, the training total of `ppml` and the cross-validation of the ridge line. After each size the checks of section 8 on `ppml` are printed, as in stage 3.

Then the steps of stage 4 on these files (`experiment_cutoff`): `rf` and `gbm` tuned on this experiment, the gaps of `ppml` against each, the pattern of each comparison, and the bootstrap. Its resamples use the keys of the base run (split 0), so resample b picks the same draws in both.

In [ ]:
runs_noise, components_noise = run_experiment(base["X_pool"], base["y_pool"], base["pair_pool"], base["X_test"],
                                              base["y_test"], "noise", DEV_YEAR, BASE_SPLIT, SIZES, DRAWS, noise=True,
                                              report_ridge=False)
noise = experiment_cutoff(runs_noise, components_noise, DEV_YEAR, BASE_SPLIT, BOOTSTRAP)
print(f"noise: {len(runs_noise):,} rows of runs, {len(components_noise):,} rows of components, {BOOTSTRAP:,} resamples")

### `noise` next to the base: settings, gaps and cutoff

First, the tuned `rf` and `gbm` at each size, without and with noise. Then, for each size and comparison (`ppml` against tuned `rf`, against tuned `gbm`, and against the better of the two, `best`), the error gap E(ppml) - E(c), the bias gap and the variance gap, without and with noise. The bias and variance gaps are the version `rn` of section 7, corrected for the number of draws and for the overlap of samples drawn from one pool, the version stage 4 prints. A negative error gap means `ppml` has the lower mean test deviance. A positive variance gap means the ensemble's predictions vary more from sample to sample than those of `ppml`. In each row, error gap = bias gap - variance gap. Last, the pattern of each comparison over the sizes (`find_crossing`, read with decision 12) with n* or `last_crossing`, and the share of bootstrap resamples in each pattern.

In [ ]:
def beside_base(base_table, check_table, check, keys, columns):
    # Each of columns twice, the base run first, on the rows matched by keys.
    merged = base_table[keys + columns].merge(check_table[keys + columns], on=keys, suffixes=("_base", f"_{check}"),
                                              validate="one_to_one")
    assert len(merged) == len(base_table) == len(check_table), f"The base run and {check} do not have the same rows."
    return merged[keys + [f"{column}_{variant}" for column in columns for variant in ("base", check)]]


def cutoffs_beside_base(check_cutoff, check):
    # The cutoff of the base run and of the check, one row per comparison and variant.
    rows = []
    for comparison in COMPARISONS:
        rows.append(cutoff_base[cutoff_base["comparison"] == comparison].assign(variant="base"))
        rows.append(check_cutoff[check_cutoff["comparison"] == comparison].assign(variant=check))
    return pd.concat(rows, ignore_index=True)


SHARES = [f"share_{pattern}" for pattern in PATTERNS]
INTERVALS = ["resamples_one_crossing", "n_star_low", "n_star_high", "resamples_reversal_then_crossing",
             "last_crossing_low", "last_crossing_high"]
ensembles_base = tuned_base[tuned_base["model"] != "ppml"]
ensembles_noise = noise["tuned"][noise["tuned"]["model"] != "ppml"]
noise_cutoffs = cutoffs_beside_base(noise["cutoff"], "noise")
with pd.option_context("display.width", 250, "display.max_columns", None):
    print("Tuned rf and gbm:")
    print(beside_base(ensembles_base, ensembles_noise, "noise", ["size", "model"], ["config"]).to_string(index=False))
    print()
    print("Gaps of ppml (bias and variance gaps: version rn):")
    print(beside_base(gaps_base, noise["gaps"], "noise", ["size", "comparison"],
                      ["model", "error_gap", "bias_gap_rn", "variance_gap_rn"])
          .to_string(index=False, float_format="{:.4g}".format))
    print()
    print("Cutoff:")
    print(noise_cutoffs[["comparison", "variant", "pattern", "n_star", "last_crossing", "crossing_from", "crossing_to"]]
          .to_string(index=False, float_format="{:.1f}".format))
    print()
    print(f"Share of the {BOOTSTRAP:,} resamples in each pattern:")
    print(noise_cutoffs[["comparison", "variant"] + SHARES].to_string(index=False, float_format="{:.3f}".format))
    print()
    print(noise_cutoffs[["comparison", "variant"] + INTERVALS].to_string(index=False, float_format="{:.1f}".format))

## The check `split2`

The base experiment of stage 3 on split 1 (`run_experiment`): another test set, another training pool, and training samples and seeds keyed by split 1. Then the steps of stage 4 on its files (`experiment_cutoff`), with the bootstrap keyed by split 1.

In [ ]:
second = splits[SECOND_SPLIT]
runs_split2, components_split2 = run_experiment(second["X_pool"], second["y_pool"], second["pair_pool"],
                                                second["X_test"], second["y_test"], "split2", DEV_YEAR, SECOND_SPLIT,
                                                SIZES, DRAWS, report_ridge=False)
split2 = experiment_cutoff(runs_split2, components_split2, DEV_YEAR, SECOND_SPLIT, BOOTSTRAP)
print(f"split2: {len(runs_split2):,} rows of runs, {len(components_split2):,} rows of components, {BOOTSTRAP:,} resamples")

### `split2` next to the base: n*

The tuned `rf` and `gbm` at each size, then the error gap of each comparison with its paired standard error and the draws `ppml` wins, and last the pattern of each comparison with n* or `last_crossing` and the bootstrap, on split 0 (the base run) and on split 1. The two splits have different test sets, so their errors are measured on different test pairs: the patterns and the sizes where the gap changes sign can be compared, the levels of the errors less so.

In [ ]:
ensembles_split2 = split2["tuned"][split2["tuned"]["model"] != "ppml"]
split2_cutoffs = cutoffs_beside_base(split2["cutoff"], "split2")
with pd.option_context("display.width", 250, "display.max_columns", None):
    print("Tuned rf and gbm:")
    print(beside_base(ensembles_base, ensembles_split2, "split2", ["size", "model"], ["config"]).to_string(index=False))
    print()
    print("Error gap of ppml:")
    print(beside_base(gaps_base, split2["gaps"], "split2", ["size", "comparison"],
                      ["model", "error_gap", "se_error_gap", "ppml_wins"])
          .to_string(index=False, float_format="{:.4g}".format))
    print()
    print("Cutoff:")
    print(split2_cutoffs[["comparison", "variant", "pattern", "n_star", "last_crossing", "crossing_from", "crossing_to"]]
          .to_string(index=False, float_format="{:.1f}".format))
    print()
    print(f"Share of the {BOOTSTRAP:,} resamples in each pattern:")
    print(split2_cutoffs[["comparison", "variant"] + SHARES].to_string(index=False, float_format="{:.3f}".format))
    print()
    print(split2_cutoffs[["comparison", "variant"] + INTERVALS].to_string(index=False, float_format="{:.1f}".format))

## The check `trees2x`

At the largest size only, on the training samples of the base run with their seeds (`run_trees2x_draw`), at the settings that `size_rule.csv` gives for that size:

- `rf` at its tuned setting with twice `RF_TREES` trees. scikit-learn draws the seed of each tree in turn from the one seed of the forest, so the first `RF_TREES` trees of this forest are the trees of the base forest. The forest is scored twice: with those first trees alone, which is the base forest again, and with all its trees.
- `gbm` at its tuned depth with twice `GBM_MAX_TREES` trees, scored through staged prediction at its tuned number of trees (the base model again), at twice that number and at the new maximum (`trees2x_counts`; when the tuned number is already the largest of the grid, the last two are the same).

The two rows that are base models again must equal the tuned rows of `dev_runs.csv` on every draw, within a relative difference of 1e-9: a check that the samples and seeds are those of the base run. For each configuration, the table gives its mean test deviance (`mean`), that of the tuned configuration on the same draws (`base_mean`), the mean of the paired difference on each draw (`difference`; negative means the extra trees lowered the error) and its paired standard error. The seconds of each row are those of the fit it came from.

In [ ]:
counts = trees2x_counts(settings["gbm"]["gbm_trees"])
print(f"n = {largest:,}: rf {settings['rf']['config']} with {2 * RF_TREES} trees, scored with its first {RF_TREES} and "
      f"with all {2 * RF_TREES}; gbm at depth {settings['gbm']['gbm_depth']} with {2 * GBM_MAX_TREES} trees, scored at "
      f"{', '.join(str(trees) for trees in counts)} trees")
groups = test_groups(base["y_test"])
rows = []
for draw, result in enumerate(run_trees2x_draws(base["X_pool"], base["y_pool"], base["X_test"], DEV_YEAR, BASE_SPLIT,
                                                largest, DRAWS, settings["rf"], settings["gbm"])):
    keys = {"variant": "trees2x", "year": DEV_YEAR, "split": BASE_SPLIT, "size": largest, "draw": draw}
    rows.extend(trees2x_rows(result, base["y_test"], groups, keys))
runs_trees2x = runs_table(rows)

differences = trees2x_differences(runs_trees2x, runs_base, rule[rule["size"] == largest])
again = differences[differences["config"] == differences["base_config"]]
assert len(again) == 2 and (again["largest_relative_difference"] <= 1e-9).all(), \
    "The rows of trees2x at the tuned settings differ from dev_runs.csv. Stop and report."
with pd.option_context("display.width", 250, "display.max_columns", None):
    print(differences.to_string(index=False, float_format="{:.4g}".format))
print(f"The tuned rf and gbm come back on all {DRAWS} draws, largest relative difference from dev_runs.csv "
      f"{again['largest_relative_difference'].max():.1e}")

## Write the three files

`check_runs.csv` has the rows of the three checks, `check_components.csv` and `check_cutoff.csv` those of `noise` and `split2`. Stage 7 computes the tuned settings and the gaps of each check from the first two with the functions of stage 4, as `experiment_cutoff` does here.

In [ ]:
check_runs = pd.concat([runs_noise, runs_split2, runs_trees2x], ignore_index=True)
check_components = pd.concat([components_noise, components_split2], ignore_index=True)
check_cutoff = pd.concat([noise["cutoff"].assign(variant="noise"), split2["cutoff"].assign(variant="split2")],
                         ignore_index=True)
check_cutoff = check_cutoff[["variant"] + list(noise["cutoff"].columns)]
assert list(check_runs["variant"].unique()) == list(CHECKS), "check_runs.csv does not hold the three checks in order."

RESULTS.mkdir(exist_ok=True)
for table, name in [(check_runs, "check_runs.csv"), (check_components, "check_components.csv"),
                    (check_cutoff, "check_cutoff.csv")]:
    table.to_csv(RESULTS / name, index=False, lineterminator="\n")
    print(f"Wrote {(RESULTS / name).relative_to(ROOT)}: {len(table):,} rows")